<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Build a test set that represents your users

**[Build a test set that represents your users](https://learning.nextia-ai.com/courses/search/m02/build-a-test-set-that-represents-your-users/)** · Search and Ranking: Find the Best Match · Module 2, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare the test set with the query log, compute a bootstrap interval by hand, compare two runs on the same queries, and find how many queries a difference needs.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 3 MB). No account, no key and no model download. |
| **You should know** | NDCG and the other measures, from [Ranking measures](https://learning.nextia-ai.com/courses/search/m02/ranking-measures/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m02/build-a-test-set-that-represents-your-users/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M02-L03-build-a-test-set-that-represents-your-users/build-a-test-set-that-represents-your-users-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 2 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/log/searches.csv.gz": "73da1fb11cebb3891cabca5ab320be2147d1855b9f8e53890007b19fb78862ea",
    "data/recorded/rerank.csv.gz": "38971117046aefd883c159ed9a7b196a0ca88f54757a8dc35df956d39222da88",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Who searches for what

The query log is **synthetic**: our code made it from WANDS queries with a click model. The 40 most searched texts are the **head**, the next 120 the **torso**, and all the others the **tail**. Run the next cell: it compares the share of the log's searches with the share of the test queries in each band.

In [ ]:
q = queries()
log = pd.read_csv("data/log/searches.csv.gz")
counts = log["query"].value_counts()           # how often each text was searched (synthetic log)
share = {"head": counts.iloc[:40].sum(), "torso": counts.iloc[40:160].sum(), "tail": counts.iloc[160:].sum()}
test = q[q["split"] == "test"]
bands = pd.DataFrame({
    "share of searches": pd.Series(share) / len(log),
    "test queries": test["band"].value_counts(),
}).loc[["head", "torso", "tail"]]
bands["share of test queries"] = bands["test queries"] / len(test)
bands.round(3)

> **Check.** The head is 0.371 of the searches but only 0.07 of the test queries. A mean over test queries gives every query the same weight, so it speaks mostly for the tail. That is a choice: say it when you report a number.

## 2. Two runs on the same test queries

Run **A** is hybrid search (Module 5). Run **B** takes A's top 50 and re-orders them with a slower model (Module 6). Both are recorded, so this notebook needs no model.

In [ ]:
j = judgments()
rr = pd.read_csv("data/recorded/rerank.csv.gz")      # the top 100 of run A, with run B's recorded scores
codes = set(q.loc[q["source"] == "code", "query_id"])

def run_a(ids):
    return {qid: list(g.sort_values("hybrid_rank")["product_id"]) for qid, g in rr[rr["query_id"].isin(ids)].groupby("query_id")}

def run_b(ids, n=50):
    out = {}
    for qid, ranked in run_a(ids).items():
        if qid in codes:                               # a product code keeps its exact product first
            out[qid] = ranked
            continue
        head = rr[(rr["query_id"] == qid) & rr["product_id"].isin(ranked[:n])]
        head = head.sort_values(["score", "product_id"], ascending=[False, True])
        out[qid] = list(head["product_id"]) + ranked[n:]
    return out

ids = set(test["query_id"])
a = cp.per_query(run_a(ids), j)
b = cp.per_query(run_b(ids), j)
print("A", round(a["ndcg@10"].mean(), 3), " B", round(b["ndcg@10"].mean(), 3))

## 3. Your turn: a bootstrap interval

A mean over 243 queries is not exact: another 243 queries would give another mean. The **bootstrap** estimates how much it could move. Draw 243 queries **with replacement** (a query can come twice), take the mean, repeat 2,000 times, and keep the middle 95% of the means. Complete `bootstrap_interval`.

In [ ]:
def bootstrap_interval(values, n=2000, seed=0):
    """The mean and its 95% bootstrap interval: draw len(values) values with replacement, n times."""
    values = np.asarray(values, float)
    values = values[~np.isnan(values)]
    rng = np.random.default_rng(seed)
    means = [rng.choice(values, len(values), replace=True).mean() for _ in range(n)]
    return values.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

mean, low, high = bootstrap_interval(a["ndcg@10"])
print(f"A: NDCG@10 {mean:.3f} (95% interval {low:.3f} to {high:.3f})")
print("compare.py:", [round(x, 3) for x in cp.bootstrap(a["ndcg@10"].to_numpy())])

In [ ]:
expect('low', low, 0.578, tolerance=0.002)

> **Check.** A's NDCG@10 is 0.621 (95% interval 0.578 to 0.665). The course's `compare.py` does the same with its own random draws, so its interval can differ in the third decimal.

## 4. Is B really better? Compare on the same queries

> **Predict.** B's mean is 0.635, A's is 0.621. Is B clearly better?

Two intervals that overlap do not answer it. Compare **pairs**: the difference on each query, then the bootstrap interval of the mean difference.

In [ ]:
d = cp.paired(a, b)        # B minus A on the same queries
print(f"B minus A: {d['difference']:+.3f} (95% interval {d['low']:+.3f} to {d['high']:+.3f}); "
      f"better on {d['better']}, worse on {d['worse']}, the same on {d['same']}")
lang = test.set_index("query_id")["language"]
rows = []
for value in ("en", "fr"):
    s = [i for i in a.index if lang[i] == value]
    r = cp.paired(a.loc[s], b.loc[s])
    rows.append({"language": value, "queries": r["queries"], "difference": r["difference"], "low": r["low"], "high": r["high"]})
by_language = pd.DataFrame(rows).round(3)
by_language

> **Check.** B minus A is +0.014 (95% interval -0.008 to +0.035). The interval contains 0, so the test set cannot say that B is better. The slices tell more: English +0.034, French -0.048. The slower model reads English only, and it hurts French queries.

## 5. Your turn: how many queries do you need?

Draw smaller test sets from the English queries (with replacement) and count how often the interval shows the difference clearly (it does not contain 0).

In [ ]:
diff = (b["ndcg@10"] - a["ndcg@10"])[[i for i in a.index if lang[i] == "en"]].dropna()
rng = np.random.default_rng(38)
clear = {}
for n in (25, 50, 100, 200, 400):
    hits = 0
    for draw in range(200):
        sub = rng.choice(diff.to_numpy(), n, replace=True)
        _, lo, hi = cp.bootstrap(sub, n=500, seed=draw)
        hits += lo > 0 or hi < 0
    clear[n] = hits / 200
print(f"English difference on all {len(diff)} queries: {diff.mean():+.3f}")
print("share of random test sets that show it clearly:", clear)

In [ ]:
smallest_n = min(n for n, share in clear.items() if share >= 0.8) if max(clear.values()) >= 0.8 else None
print(smallest_n)

In [ ]:
expect('smallest_n', smallest_n, 200)

> **Check.** With 25 queries, the difference is clear in 0.24 of the draws; with 100, in 0.685. You need about 200 queries to see it in 8 draws of 10. A small test set misses a real difference most of the time. Decide the size of the test set from the smallest difference that matters to you.

## 6. In the project

```sh
search eval --run bm25 --by band        # NDCG@10 per band, each with its interval
search compare current bm25 --by band   # the paired difference, overall and per band
```

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Judgments from a language model](https://learning.nextia-ai.com/courses/search/m02/judgments-from-a-language-model/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m02/build-a-test-set-that-represents-your-users/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.